# Step 10c — MVT Grid: MiniImageNet, both shots

**Settings:** GPU T4, Internet ON, attach the Kaggle dataset `beft-thesis-data`
(owner `notavailable73` — note the slug is missing the "p" in "bpeft", a
pre-existing typo in how the dataset was created; the display title still
reads "bpeft-thesis-data"). See `step_writeups/step10.txt` for the full
reasoning and `plan.md` for the grid design. This notebook only drives
`scripts/run_mvt_grid.py --only "dataset=mini_imagenet"`; every other Step 10 script (config
generation, aggregation, tables, plots) is shared across all three notebooks
and documented there — nothing new lives in this notebook itself.


## 1. GPU check + clone repo + install deps

In [1]:
import torch, sys, os, subprocess
print('python:', sys.version.split()[0], '| torch:', torch.__version__)
print('cuda  :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU T4'

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/thesis'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repo ready at', os.getcwd())


python: 3.12.13 | torch: 2.10.0+cu128
cuda  : True | Tesla T4


Cloning into '/kaggle/working/thesis'...


repo ready at /kaggle/working/thesis


## 2. Stage data — symlink the attached dataset into `data/`

Nothing here is a hard requirement: every dataset falls back to a runtime download if its symlink source is missing (Internet must be ON).

In [2]:
# Attach the Kaggle dataset `beft-thesis-data` (owner notavailable73) before
# running this cell: https://www.kaggle.com/datasets/notavailable73/beft-thesis-data
# Its real upload structure (verified via the Kaggle API, 2026-08-03) is:
#   bpeft-data/cifar-100-python/{meta,train,test}
#   bpeft-data/svhn/test_32x32.mat
#   bpeft-data/tinyimagenet/tiny-imagenet-200/tiny-imagenet-200/{train,val,test,wnids.txt,...}
#   bpeft-data/miniimagenet/mini-imagenet-cache-{train,validation,test}.pkl
# (Zenodo pkl caches, not the .npy/.json format plan.md Section 4.2 originally
# assumed -- src/datasets/mini_imagenet.py already supports this layout
# natively, so nothing needed regenerating.) Rather than hardcode that exact
# nesting, this reuses the SAME staged-path finder functions
# scripts/train.py / evaluate.py call at runtime, so it is guaranteed to
# symlink to whatever those modules would discover themselves -- correct
# regardless of whether Kaggle mounts this dataset one level deeper
# (`/kaggle/input/datasets/<owner>/<slug>/...`) or double-wraps the
# tiny-imagenet-200 folder on auto-unzip (both observed live in earlier
# steps; see each finder's docstring).
import os, shutil

from src.datasets.cifar_fs import _find_staged_cifar100_root
from src.datasets.svhn_ood import _find_staged_svhn_root
from src.datasets.tinyimagenet_ood import _find_extracted_tin_root
from src.datasets.mini_imagenet import _find_zenodo_pkls

LINKS = {}

cifar100_root = _find_staged_cifar100_root('data')
if cifar100_root:
    LINKS['data/cifar-100-python'] = os.path.join(cifar100_root, 'cifar-100-python')

svhn_root = _find_staged_svhn_root('data')
if svhn_root:
    LINKS['data/svhn/test_32x32.mat'] = os.path.join(svhn_root, 'test_32x32.mat')

tin_root = _find_extracted_tin_root('data')
if tin_root:
    LINKS['data/tiny-imagenet-200'] = tin_root

for split, pkl_path in (_find_zenodo_pkls('data') or {}).items():
    LINKS[f'data/{pkl_path.name}'] = str(pkl_path)

os.makedirs('data/svhn', exist_ok=True)
if not LINKS:
    print('No staged files found under /kaggle/input -- did you attach '
          '`beft-thesis-data`? Falling back to runtime downloads for everything.')
for link, target in LINKS.items():
    if os.path.exists(link) or os.path.islink(link):
        print(f'OK   (already present): {link}')
        continue
    if not os.path.exists(target):
        print(f'MISSING source -- {os.path.basename(link)} will fall back to '
             f'runtime download (target not found: {target})')
        continue
    try:
        os.symlink(target, link)
        print(f'OK   (symlinked): {link} -> {target}')
    except OSError as e:
        print(f'symlink failed ({e}); copying instead (slower): {link}')
        (shutil.copytree if os.path.isdir(target) else shutil.copy2)(target, link)
        print(f'OK   (copied): {link}')


OK   (symlinked): data/cifar-100-python -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/cifar-100-python
OK   (symlinked): data/svhn/test_32x32.mat -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/svhn/test_32x32.mat
OK   (symlinked): data/tiny-imagenet-200 -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/tinyimagenet/tiny-imagenet-200/tiny-imagenet-200
OK   (symlinked): data/mini-imagenet-cache-train.pkl -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/miniimagenet/mini-imagenet-cache-train.pkl
OK   (symlinked): data/mini-imagenet-cache-validation.pkl -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/miniimagenet/mini-imagenet-cache-validation.pkl
OK   (symlinked): data/mini-imagenet-cache-test.pkl -> /kaggle/input/datasets/notavailable73/beft-thesis-data/bpeft-data/miniimagenet/mini-imagenet-cache-test.pkl


## 3. Build the frozen splits (CIFAR-FS + MiniImageNet)

In [3]:
!python scripts/build_cifar_fs_split.py
!python scripts/build_mini_imagenet_split.py


wrote /kaggle/working/thesis/data/cifar_fs_split.json  (64/16/20, disjoint, union=100, status=canonical_bertinetto_via_torchmeta)
wrote /kaggle/working/thesis/data/mini_imagenet_split.json  (64/16/20, disjoint, union=100, status=canonical_ravi_larochelle)


## 4. Generate the 120 grid configs + run the offline config tests

In [4]:
!python scripts/build_grid_configs.py
!python -m pytest -q tests/test_grid_configs.py


wrote 120 configs to /kaggle/working/thesis/configs/grid/ (96 PEFT + 24 baseline)
wrote /kaggle/working/thesis/configs/grid/_index.json
  priority 1: 36 cells
  priority 2: 24 cells
  priority 3: 36 cells
  priority 4: 24 cells
...................                                                      [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-pa

### 4b. MiniImageNet cache sanity check (this notebook is the one that exercises it most)

In [5]:
# Warms the .npy caches every mini_imagenet grid cell below needs (train for
# meta-training, val for validation + the near-OOD pool, test for episodes) and
# asserts the frozen Ravi & Larochelle split sizes -- condensed from
# notebooks/step9-mini.ipynb Section 5, which is where this pattern and the
# 64/16/20 expectations come from. get_mini_imagenet returns a Dataset, NOT an
# (x, y) tuple, so counts come from len(ds) / ds.wnids and the per-image shape
# from ds[0][0]; a prior version unpacked the return value as `x, y` and died
# with "too many values to unpack" on the train split.
import time

from src.datasets.mini_imagenet import get_mini_imagenet, MINI_IMAGENET_ALL_WNIDS

t0 = time.time()
splits = {}
for split in ('train', 'val', 'test'):
    ds = get_mini_imagenet(data_root='data', image_size=224, split=split)
    splits[split] = ds
    print(f'[{split}] {len(ds)} images, {len(ds.wnids)} classes, '
          f'x={tuple(ds[0][0].shape)} ({time.time() - t0:.0f}s elapsed)')

assert len(splits['train'].wnids) == 64
assert len(splits['val'].wnids) == 16
assert len(splits['test'].wnids) == 20
assert (set(splits['train'].wnids) | set(splits['val'].wnids)
        | set(splits['test'].wnids)) == MINI_IMAGENET_ALL_WNIDS
print(f'\ncache build: {time.time() - t0:.0f}s -- split sizes + disjointness OK')


[train] 38400 images, 64 classes, x=(3, 224, 224) (12s elapsed)
[val] 9600 images, 16 classes, x=(3, 224, 224) (15s elapsed)
[test] 12000 images, 20 classes, x=(3, 224, 224) (19s elapsed)

cache build: 19s -- split sizes + disjointness OK


## 5. Log in to W&B + run the grid

Add a Kaggle Secret named `WANDB_API_KEY` (notebook editor → Add-ons →
Secrets; get the key from <https://wandb.ai/authorize>) before running this
cell so the grid's runs upload online and group by (dataset, shots) per
`progress.txt`'s Step 10 exit criteria. Falls back to **offline** mode
(writes to `./wandb/`, sync later with `wandb sync wandb/`) if the secret is
missing or login fails — this cell never calls interactive `wandb.login()`,
so the unattended `--max-minutes 660` run below can't stall waiting on
stdin. Login and the grid launch are ONE cell on purpose (a prior version
split them across two cells and passed `WANDB_MODE` to a separate `!`
shell cell via `--wandb-mode {WANDB_MODE}`; that silently broke if the two
cells were ever run out of order or after a kernel restart, since IPython
leaves an unresolved `{name}` in a `!` command as literal text instead of
erroring). Resumable: re-running this cell after a session timeout picks up
where it left off (`--resume` skips any cell whose results JSON already
exists).

In [6]:
import os
import subprocess
import sys

import wandb

api_key = os.environ.get("WANDB_API_KEY")
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if api_key:
            os.environ["WANDB_API_KEY"] = api_key
            print("Loaded WANDB_API_KEY from Kaggle Secrets.")
    except Exception as e:
        print(f"Kaggle Secrets lookup skipped: {e!r}")

if not api_key:
    print("No WANDB_API_KEY found (env or Kaggle Secrets) -- using offline "
          "mode so this run never blocks on an interactive login prompt.")
    WANDB_MODE = "offline"
else:
    WANDB_MODE = "online"
    try:
        if not wandb.login(key=api_key):
            print("wandb.login() returned False -- falling back to offline mode.")
            WANDB_MODE = "offline"
    except Exception as e:
        print(f"wandb.login() failed: {e!r} -- falling back to offline mode.")
        WANDB_MODE = "offline"

print("WANDB_MODE for this session:", WANDB_MODE)

cmd = [sys.executable, "scripts/run_mvt_grid.py",
       "--resume", "--only", "dataset=mini_imagenet", "--priority",
       "--max-minutes", "660", "--use-tinyimagenet", "--use-gaussian",
       "--wandb-mode", WANDB_MODE]
print(">>>", " ".join(cmd))
subprocess.run(cmd)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Loaded WANDB_API_KEY from Kaggle Secrets.


wandb: Currently logged in as: fatpotato (fatpotato-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WANDB_MODE for this session: online
>>> /usr/bin/python3 scripts/run_mvt_grid.py --resume --only dataset=mini_imagenet --priority --max-minutes 660 --use-tinyimagenet --use-gaussian --wandb-mode online
[grid] 48 cells selected (filtered by dataset=mini_imagenet)
[grid] (1/48) mini_imagenet/5shot/mobilenetv3_small/bottleneck_parallel/evidential/seed42
[03:28:58] INFO bpeft.train: config: /kaggle/working/thesis/configs/grid/mini_5shot_mbnet_parallel_evidential_seed42.yaml  seed: 42  trainer.type: episodic


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: fatpotato (fatpotato-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/thesis/wandb/run-20260804_032859-0txpjqz8
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mobilenetv3_small_bottleneck_prototype_mini_imagenet_5shot_seed42
wandb: ⭐️ View project at https://wandb.ai/fatpotato-personal/bpeft-thesis
wandb: 🚀 View run at https://wandb.ai/fatpotato-personal/bpeft-thesis/runs/0txpjqz8


...

wandb: setting up run qparvohp
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/thesis/wandb/run-20260804_132952-qparvohp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet18_lora_prototype_mini_imagenet_1shot_seed44_eval
wandb: ⭐️ View project at https://wandb.ai/fatpotato-personal/bpeft-thesis
wandb: 🚀 View run at https://wandb.ai/fatpotato-personal/bpeft-thesis/runs/qparvohp


[13:29:54] INFO bpeft.evaluate: wandb run: resnet18_lora_prototype_mini_imagenet_1shot_seed44_eval  url=https://wandb.ai/fatpotato-personal/bpeft-thesis/runs/qparvohp
[13:29:54] INFO bpeft.evaluate: loaded checkpoint: checkpoints/model_phase2_grid_mini_1shot_r18_lora_prototype-softmax_seed44.pt  best_val_epoch=2  best_val_acc=0.792
[13:34:09] INFO bpeft.evaluate: OOD pools: svhn_far=(500, 512), mini_near=(500, 512), tin_near=(500, 512), gaussian_far=(500, 512)
[13:34:30] INFO bpeft.evaluate: fit temperature on 100 val episodes: T=0.5166
[13:34:30] INFO bpeft.evaluate: ep   0  acc=0.867  F1=0.869  ECE=0.236  Brier=0.285  AUROC[svhn_far/msp]=0.934
[13:34:30] INFO bpeft.evaluate: ep   1  acc=0.613  F1=0.571  ECE=0.125  Brier=0.540  AUROC[svhn_far/msp]=0.659
[13:34:30] INFO bpeft.evaluate: ep   2  acc=0.800  F1=0.786  ECE=0.269  Brier=0.386  AUROC[svhn_far/msp]=0.631
[13:34:31] INFO bpeft.evaluate: ep   3  acc=0.853  F1=0.854  ECE=0.263  Brier=0.352  AUROC[svhn_far/msp]=0.783
[13:34:31] IN

wandb: updating run metadata; uploading history steps 497-564, summary, console lines 501-568; uploading artifact metrics_grid_mini_1shot_r18_lora_seed44_lora_prototype-softmax
wandb: uploading history steps 497-564, summary, console lines 501-568; uploading artifact metrics_grid_mini_1shot_r18_lora_seed44_lora_prototype-softmax
wandb: uploading history steps 497-564, summary, console lines 501-568
wandb: uploading history steps 497-564, summary, console lines 501-568; uploading wandb-summary.json; uploading config.yaml; uploading media/images/plots/reliability_diagram_600_739bc41fb073f79765bb.png; uploading media/images/plots/confusion_matrix_602_4d98990cfe9d02ac0c60.png (+ 1 more)
wandb: uploading history steps 497-564, summary, console lines 501-568
wandb: uploading history steps 565-602, summary, console lines 569-604
wandb: uploading summary
wandb: 
wandb: Run history:
wandb:                   eval/ECE ▇▆▅▇▇▅▅▄▃▄▃▆▆▇▃▅▇▇▅▆▃▄▆▅▅▁▅▆▅▂▅▃▅▃▆█▆▅▃▅
wandb:              eval/accuracy ▅▂▅▆

{
  "accuracy_ci95": 0.0073460116986620114,
  "accuracy_mean": 0.8055333516001701,
  "accuracy_std": 0.09180602197060117,
  "adapter_type": "lora",
  "best_val_epoch": 2,
  "brier_mean": 0.3283783451343576,
  "brier_std": 0.08898154506068194,
  "brier_ts": 0.2782880961894989,
  "config_path": "/kaggle/working/thesis/configs/grid/mini_1shot_r18_lora_softmax_seed44.yaml",
  "ece_per_episode_mean": 0.2084801755766074,
  "ece_per_episode_std": 0.05285808287049181,
  "ece_pooled": 0.18158925670319132,
  "ece_ts": 0.010311360302236344,
  "episodes_file": "configs/test_episodes.yaml",
  "f1_macro_ci95": 0.008119900683603163,
  "f1_macro_mean": 0.7966246933803663,
  "f1_macro_std": 0.10147761957604114,
  "fpr_at_95_tpr__gaussian_far__energy": 0.19508333333333333,
  "fpr_at_95_tpr__gaussian_far__msp": 0.8223133333333333,
  "fpr_at_95_tpr__gaussian_far__ts_msp": 0.8416266666666665,
  "fpr_at_95_tpr__mini_near__energy": 0.6652433333333334,
  "fpr_at_95_tpr__mini_near__msp": 0.7760633333333333,
  

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_mvt_grid.py', '--resume', '--only', 'dataset=mini_imagenet', '--priority', '--max-minutes', '660', '--use-tinyimagenet', '--use-gaussian', '--wandb-mode', 'online'], returncode=0)

## 6. Pack + push this notebook's slice of results

In [7]:
# Pack + push this notebook's slice only (mirrors Step 9 notebook Section 9/9b).
import glob, hashlib, json as _json, os, subprocess, zipfile

ARTIFACT_STEM = 'step10c_mini'
RUN_TAG_GLOB = 'grid_mini'

ZIP = f'/kaggle/working/{ARTIFACT_STEM}_artifacts.zip'
RESULTS = sorted(glob.glob(f'results/grid/*{RUN_TAG_GLOB}*'))
LOGS = ['results/grid/_run_log.jsonl'] if os.path.exists('results/grid/_run_log.jsonl') else []
CHECKPOINTS = sorted(glob.glob(f'checkpoints/model_phase2_{RUN_TAG_GLOB}*.pt'))
ALL_FILES = RESULTS + LOGS + CHECKPOINTS

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in ALL_FILES:
        zf.write(p)
    manifest_lines = [
        f'{p}  {os.path.getsize(p)}B  sha256={hashlib.sha256(open(p, "rb").read()).hexdigest()}'
        for p in ALL_FILES
    ]
    zf.writestr('MANIFEST.txt', '\n'.join(manifest_lines))
print(f'wrote {ZIP} ({len(ALL_FILES)} files)')

# Channel 1: push to a Kaggle dataset (survives the browser tab closing) if
# KAGGLE_USERNAME / KAGGLE_KEY secrets are configured; channel 2: browser
# download otherwise. See notebooks/step9-mini(1).ipynb Section 9b for the
# fuller version of this pattern this is condensed from.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')
    HAVE_SECRETS = True
except Exception:
    HAVE_SECRETS = False

if HAVE_SECRETS:
    ds_dir = f'/kaggle/working/{ARTIFACT_STEM}_dataset'
    os.makedirs(ds_dir, exist_ok=True)
    subprocess.run(['cp', ZIP, ds_dir], check=True)
    meta = {
        "title": f"{ARTIFACT_STEM}-artifacts",
        "id": f"{os.environ['KAGGLE_USERNAME']}/{ARTIFACT_STEM}-artifacts",
        "licenses": [{"name": "CC0-1.0"}],
    }
    _json.dump(meta, open(f'{ds_dir}/dataset-metadata.json', 'w'))
    r = subprocess.run(['kaggle', 'datasets', 'create', '-p', ds_dir, '-q'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run(['kaggle', 'datasets', 'version', '-p', ds_dir, '-m', 'update', '-q'])
    print(f'pushed to Kaggle dataset {ARTIFACT_STEM}-artifacts')
else:
    from IPython.display import FileLink, display
    display(FileLink(ZIP))
    print('no Kaggle Secrets found -- use the download link above instead.')


wrote /kaggle/working/step10c_mini_artifacts.zip (241 files)
pushed to Kaggle dataset step10c_mini-artifacts


## After this session

1. Do **not** run `scripts/aggregate_grid.py` / `make_master_tables.py` /
   `grid_plots.py` here — those run ONCE, locally or on plain CPU, after
   10a + 10b + 10c have ALL finished and their artifact zips are merged into
   one local `results/grid/` (plan.md Section 10.4).
2. Note any `collapsed` or `error` status lines from Section 6's output, and
   `results/grid/_run_log.jsonl`'s `best_val_epoch` values, for
   `step_writeups/step10.txt`.
